In [5]:
!pip install arch statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 25.5 MB/s eta 0:00:00


In [6]:
import warnings
from arch import arch_model
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

# CẤU HÌNH HỆ THỐNG VÀ DANH SÁCH TẬP DỮ LIỆU CHẠY 3 NGÂN HÀNG
datasets = [
    {"ticker": "JPM", "filename": "JPM_master_arimax.csv"},
    {"ticker": "WFC", "filename": "WFC_master_arimax.csv"},
    {"ticker": "BAC", "filename": "BAC_master_arimax.csv"},
]

target_col = "target_return"
exog_cols = [
    "lag_return_1",
    "SPX_return_lag1",
    "VIX_return_lag1",
    "EFFR_change_lag1",
    "Yield_Spread_diff1_lag1",
]
test_years = [2023, 2024, 2025]

# Thùng chứa kết quả Kinh tế lượng toàn cục phục vụ cho bước so sánh cuối cùng
econometric_master_results = {}

print("=== BẮT ĐẦU CHẠY PIPELINE MÔ HÌNH KINH TẾ LƯỢNG (ECONOMETRIC BASELINES) ===")

# =====================================================================
# VÒNG LẶP CHÍNH DUYỆT QUA TỪNG NGÂN HÀNG
# =====================================================================
for dataset in datasets:
    ticker = dataset["ticker"]
    filename = dataset["filename"]

    print("\n" + "="*90)
    print(f" ⏳ KHỞI CHẠY KHỐI KINH TẾ LƯỢNG CHO NGÂN HÀNG: {ticker} (File: {filename})")
    print("="*90)

    try:
        # 1. Đọc và tiền xử lý dữ liệu chuỗi thời gian dùng chung
        df_econ = pd.read_csv(filename)
        df_econ["Date"] = pd.to_datetime(df_econ["Date"])
        df_econ = df_econ.sort_values("Date").reset_index(drop=True)
    except FileNotFoundError:
        print(f" Không tìm thấy file {filename}. Bỏ qua ngân hàng {ticker}.")
        continue

    # Thiết lập cấu trúc lưu trữ cục bộ cho ngân hàng hiện tại
    results = {
        "ARIMAX": {"preds": [], "actuals": []},
        "ARMA-GARCH": {"preds": [], "actuals": [], "vols": []},
    }

    years = df_econ["Date"].dt.year

    # 2. VÒNG LẶP WALK-FORWARD VALIDATION TỪNG NĂM
    for test_yr in test_years:
        print(f"  ▶ Đang xử lý khối kiểm thử dữ liệu năm: {test_yr}")

        train_mask = (years >= 2020) & (years < test_yr)
        test_mask = years == test_yr

        df_train_base = df_econ.loc[train_mask].reset_index(drop=True)
        df_test_year = df_econ.loc[test_mask].reset_index(drop=True)

        y_train = df_train_base[target_col].values
        X_train_df = df_train_base[exog_cols]
        X_train_np = X_train_df.values

        y_test = df_test_year[target_col].values
        X_test_df = df_test_year[exog_cols]
        X_test_np = X_test_df.values

        # --- MODEL 1: ARIMAX ---
        model_arimax = ARIMA(endog=y_train, exog=X_train_np, order=(1, 0, 0))
        fit_arimax = model_arimax.fit()
        preds_arimax = fit_arimax.forecast(steps=len(y_test), exog=X_test_np)

        results["ARIMAX"]["preds"].extend(preds_arimax)
        results["ARIMAX"]["actuals"].extend(y_test)

        # Áp dụng ngưỡng động (Mean) để tính toán hướng đi thực tế cho từng Fold
        threshold_arimax = np.mean(preds_arimax)
        acc_arimax = np.mean(np.where(y_test > 0, 1, 0) == np.where(preds_arimax > threshold_arimax, 1, 0)) * 100

        # --- MODEL 2: ARMA-GARCH ---
        y_combined = pd.concat([df_train_base[target_col], df_test_year[target_col]], ignore_index=True)
        X_combined = pd.concat([X_train_df, X_test_df], ignore_index=True)
        train_len = len(df_train_base)
        test_len = len(df_test_year)

        scale_factor = 100.0
        y_combined_scaled = y_combined * scale_factor

        model_garch = arch_model(y_combined_scaled, x=X_combined, mean="ARX", lags=0, vol="GARCH", p=1, q=1, dist="StudentsT", rescale=False)
        fit_garch = model_garch.fit(last_obs=train_len, disp="off", options={"method": "L-BFGS-B", "max_iter": 1000})

        x_forecast_dict = {col: X_test_df[col].values.reshape(test_len, 1) for col in exog_cols}
        forecasts = fit_garch.forecast(start=train_len, align="target", method="analytic", x=x_forecast_dict)

        fold_garch_preds = forecasts.mean.dropna().values.flatten() / scale_factor
        fold_garch_vols = np.sqrt(forecasts.variance.dropna().values.flatten()) / scale_factor

        # Align y_test với độ dài thực tế nhận được của GARCH dự báo
        actuals_aligned = y_test[:len(fold_garch_preds)]

        results["ARMA-GARCH"]["preds"].extend(fold_garch_preds)
        results["ARMA-GARCH"]["actuals"].extend(actuals_aligned)
        results["ARMA-GARCH"]["vols"].extend(fold_garch_vols)

        threshold_garch = np.mean(fold_garch_preds)
        acc_garch = np.mean(np.where(actuals_aligned > 0, 1, 0) == np.where(fold_garch_preds > threshold_garch, 1, 0)) * 100

        print(f"    ↳ [Accuracy] ARIMAX: {acc_arimax:.2f}% | ARMA-GARCH: {acc_garch:.2f}%")
        print(f"    ↳ [Mean Vol] ARMA-GARCH: {np.mean(fold_garch_vols):.6f}")

    # Đóng gói mảng kết quả của Ngân hàng hiện tại vào Master Dictionary toàn cục
    econometric_master_results[ticker] = results

    # 3. BÁO CÁO HIỆU SUẤT TỔNG THỂ OUT-OF-SAMPLE CHO NGÂN HÀNG HIỆN TẠI
    econ_report = []
    for m_name, data in results.items():
        actuals = np.array(data["actuals"])
        preds = np.array(data["preds"])

        rmse = np.sqrt(mean_squared_error(actuals, preds))
        mae = mean_absolute_error(actuals, preds)

        # [CẢI TIẾN CHUẨN QUANT] Áp dụng ngưỡng Mean động tổng thể của mảng dự báo để loại bỏ bias thị trường một chiều
        overall_threshold = np.mean(preds)
        acc = np.mean(np.where(actuals > 0, 1, 0) == np.where(preds > overall_threshold, 1, 0)) * 100

        econ_report.append({
            "Model Baseline": m_name,
            "Overall RMSE": round(rmse, 5),
            "Overall MAE": round(mae, 5),
            "Directional Accuracy (%)": round(acc, 2)
        })

    report_df = pd.DataFrame(econ_report)
    print("\n" + "-" * 75)
    print(f"BẢNG HIỆU SUẤT TỔNG THỂ KHỐI KINH TẾ LƯỢNG - {ticker} (2023-2025)")
    print("-" * 75)
    print(report_df.to_string(index=False))
    print("-" * 75)

print("\n" + "="*90)
print("PIPELINE KINH TẾ LƯỢNG (ARIMAX & ARMA-GARCH) CHO CẢ 3 NGÂN HÀNG ĐÃ HOÀN TẤT TỰ ĐỘNG!")
print("="*90)

=== BẮT ĐẦU CHẠY PIPELINE MÔ HÌNH KINH TẾ LƯỢNG (ECONOMETRIC BASELINES) ===

 ⏳ KHỞI CHẠY KHỐI KINH TẾ LƯỢNG CHO NGÂN HÀNG: JPM (File: JPM_master_arimax.csv)
  ▶ Đang xử lý khối kiểm thử dữ liệu năm: 2023
    ↳ [Accuracy] ARIMAX: 47.60% | ARMA-GARCH: 49.80%
    ↳ [Mean Vol] ARMA-GARCH: 0.015220
  ▶ Đang xử lý khối kiểm thử dữ liệu năm: 2024
    ↳ [Accuracy] ARIMAX: 47.22% | ARMA-GARCH: 49.40%
    ↳ [Mean Vol] ARMA-GARCH: 0.014987
  ▶ Đang xử lý khối kiểm thử dữ liệu năm: 2025
    ↳ [Accuracy] ARIMAX: 53.41% | ARMA-GARCH: 48.79%
    ↳ [Mean Vol] ARMA-GARCH: 0.015946

---------------------------------------------------------------------------
BẢNG HIỆU SUẤT TỔNG THỂ KHỐI KINH TẾ LƯỢNG - JPM (2023-2025)
---------------------------------------------------------------------------
Model Baseline  Overall RMSE  Overall MAE  Directional Accuracy (%)
        ARIMAX       0.01459      0.01002                     48.87
    ARMA-GARCH       0.01457      0.00997                     49.47
----------